# Pandas 핵심 실습

위에서 아래로 실행하며 출력 결과를 확인합니다. 코드 옆 주석은 무엇을 살펴볼지 알려 줍니다.
- 강의자료와는 **N번호**로 연결합니다. Q는 확인 문제, E는 종합 실습입니다.
- 중간 셀부터 시작하면 앞에서 만든 변수가 없을 수 있습니다. 처음 실행할 때는 커널을 재시작하고 위에서부터 실행하세요.
- 노트북 옆에 `data` 폴더를 둡니다. 필요한 패키지는 `pandas`입니다.
- 기초 실습에서는 `age`, `fare`, `small`을 사용합니다. Titanic에서는 원자료 `raw`, 필요한 열을 고른 `df`, 정제 중인 `clean`을 구분합니다.
- 확인 문제는 질문을 먼저 읽고 결과를 예상한 뒤, 아래 풀이 코드를 실행합니다.

[Series와 표](#n01) · [두 방식으로 같은 표 만들기](#n04) · [선택과 조건](#n09) · [Titanic 읽기](#n12) · [복사와 중복](#n18) · [결측값](#n21) · [집계](#n26) · [종합 실습](#e01)

### 실습 준비

라이브러리는 여기서 한 번만 불러옵니다. `pd`는 pandas에 붙이는 관례적인 별칭입니다.

In [1]:
# N01
import pandas as pd
from pathlib import Path
print("Pandas 버전:", pd.__version__)

Pandas 버전: 2.3.3


<a id="n01"></a>
## N01 · Series에 값과 이름표 붙이기

Series는 인덱스 레이블이 붙은 1차원 데이터입니다. 값의 개수와 차원의 개수는 다릅니다.

### 자동 인덱스를 A·B·C로 바꾸기

값은 그대로 두고 이름표만 바꿉니다.

In [ ]:
# N01
age_default = pd.Series([22, 38, 26])
print("자동 인덱스:")
#자동으로 인덱싱이 되더라
print(age_default)
print(age_default.index)  # stop=3은 3 미만: 실제 레이블은 0, 1, 2

#인덱스를 같이 넘겨주기
age_named = pd.Series([22, 38, 26], index=["A", "B", "C"])
print("이름표를 붙인 뒤:")
print(age_named)
print("B의 나이:", age_named.loc["B"])  # 38

자동 인덱스:
0    22
1    38
2    26
dtype: int64
RangeIndex(start=0, stop=3, step=1)
이름표를 붙인 뒤:
A    22
B    38
C    26
dtype: int64
B의 나이: 38


<a id="n02"></a>
## N02 · 클래스와 자료형 구분하기

pd.Series와 pd.DataFrame은 클래스입니다. 괄호에 인수를 전달하여 객체를 만듭니다. type은 객체의 클래스, dtype(data type)은 Series 안의 값의 자료형입니다.

### 같은 값을 Series와 한 열짜리 표에 담기

N01의 Series를 다시 만들지 않고 사용합니다.

In [5]:
# N02
one_column = pd.DataFrame({"Age": age_named})
print("Series의 클래스:", type(age_named))
print("표의 클래스:", type(one_column))
print("Series 값의 자료형:", age_named.dtype)
print("Series 객체인가?", isinstance(age_named, pd.Series))
#중요 시험
print("Series 모양:", age_named.shape)      # (3,)
print("표의 모양:", one_column.shape)       # (3, 1)
print(age_named)
print(one_column)

Series의 클래스: <class 'pandas.core.series.Series'>
표의 클래스: <class 'pandas.core.frame.DataFrame'>
Series 값의 자료형: int64
Series 객체인가? True
Series 모양: (3,)
표의 모양: (3, 1)
A    22
B    38
C    26
dtype: int64
   Age
A   22
B   38
C   26


<a id="n03"></a>
## N03 · 생성자에 데이터와 설정 전달하기

data는 값, index는 각 원소의 이름표, name은 Series 전체 이름입니다. 이름을 붙여도 열 축이 새로 생기지 않습니다.

### 인덱스·이름·자료형을 각각 지정하기

이후 기초 실습에서 사용할 나이 Series를 만듭니다. 소수 요금과 구조를 비교하기 위해 float64로 통일합니다.

In [6]:
# N03
#name : 시리즈에 이름 붙이기. Data frame이 되는건 아니다.
age = pd.Series(data=[22, 38, 26], index=["A", "B", "C"],
                dtype="float64", name="Age")
print(age)
print("각 원소의 이름표:", age.index.tolist())
print("전체 이름:", age.name)
print("값의 자료형:", age.dtype)
print("기본 이름:", age_default.name)  # 이름을 생략했던 Series는 None
print("기본 자료형:", age_default.dtype)

A    22.0
B    38.0
C    26.0
Name: Age, dtype: float64
각 원소의 이름표: ['A', 'B', 'C']
전체 이름: Age
값의 자료형: float64
기본 이름: None
기본 자료형: int64


### 값 3개에 이름표 2개만 주면 어떻게 될까?

리스트로 값을 전달할 때는 값과 인덱스의 개수가 같아야 합니다. 오류를 잡아서 메시지만 보고 다음 실습으로 넘어갑니다.

In [7]:
# N03
try:
    pd.Series([22, 38, 26], index=["A", "B"])
except ValueError as error:
    print(type(error).__name__, str(error))  # 길이가 맞지 않는다는 설명

ValueError Length of values (3) does not match length of index (2)


<a id="n04"></a>
## N04 · Series를 열로 합치거나 행으로 쌓기

같은 승객 정보를 항목별 또는 사람별로 묶어 같은 표를 만듭니다. 모든 Series를 float64로 맞춰 자료형 차이가 비교를 방해하지 않도록 합니다.

### 항목별 Series 두 개를 열로 구성하기

나이 Series는 N03에서 만들었습니다. 요금 Series만 추가합니다. 딕셔너리의 키가 열 이름이 됩니다.

In [9]:
# N04
fare = pd.Series([7.25, 71.28, 7.93], index=["A", "B", "C"],
                 name="Fare", dtype="float64")
by_columns = pd.DataFrame({"Age": age, "Fare": fare})
print(by_columns)
print("행 레이블:", by_columns.index.tolist())
print("열 레이블:", by_columns.columns.tolist())
print("모양:", by_columns.shape)  # 3행 × 2열, 인덱스는 데이터 열에 포함하지 않음

    Age   Fare
A  22.0   7.25
B  38.0  71.28
C  26.0   7.93
행 레이블: ['A', 'B', 'C']
열 레이블: ['Age', 'Fare']
모양: (3, 2)


### 열 이름 하나와 열 이름의 리스트 비교하기

문자열 하나를 주면 Series, 열 이름들의 리스트를 주면 DataFrame입니다. 대괄호 수가 차원을 만드는 것이 아니라 전달한 객체의 종류에 따라 선택 방식이 달라집니다.

In [ ]:
# N04
print("이름 하나:", type(by_columns["Age"]), by_columns["Age"].shape)
print("이름의 리스트:", type(by_columns[["Age"]]), by_columns[["Age"]].shape)
#시험
#return Series
print(by_columns["Age"])    # 1차원: (3,)
#return data frame
print(by_columns[["Age"]])  # 2차원: (3, 1)

이름 하나: <class 'pandas.core.series.Series'> (3,)
이름의 리스트: <class 'pandas.core.frame.DataFrame'> (3, 1)
A    22.0
B    38.0
C    26.0
Name: Age, dtype: float64
    Age
A  22.0
B  38.0
C  26.0


### 사람별 Series 세 개를 행으로 구성하기

이번에는 Age·Fare가 원소의 인덱스 레이블이고 A·B·C가 Series 이름입니다. 이 이름들이 표의 행 인덱스로 사용됩니다.

In [11]:
# N04
person_a = pd.Series([22, 7.25], index=["Age", "Fare"], name="A", dtype="float64")
person_b = pd.Series([38, 71.28], index=["Age", "Fare"], name="B", dtype="float64")
person_c = pd.Series([26, 7.93], index=["Age", "Fare"], name="C", dtype="float64")
print(person_a)
print("값:", person_a.tolist(), "/ 개수:", person_a.size)  # 두 값이지 한 값이 아님
print("이름:", person_a.name, "/ 모양:", person_a.shape)  # (2,)

by_rows = pd.DataFrame([person_a, person_b, person_c])
print(by_rows)  # name을 사용하므로 index=["A", "B", "C"]를 다시 쓸 필요 없음
print("두 방식이 같은 표인가?", by_rows.equals(by_columns))  # True

Age     22.00
Fare     7.25
Name: A, dtype: float64
값: [22.0, 7.25] / 개수: 2
이름: A / 모양: (2,)
    Age   Fare
A  22.0   7.25
B  38.0  71.28
C  26.0   7.93
두 방식이 같은 표인가? True


### 같은 표에서 한 행과 한 열을 다시 꺼내기

한 행을 꺼내면 사람별 Series, 한 열을 꺼내면 항목별 Series가 됩니다. equals는 값과 레이블·자료형 등이 같은지 비교합니다.

In [ ]:
# N04
print("A행:")
#return Series
print(by_rows.loc["A"])
print("원래 person_a와 같은가?", by_rows.loc["A"].equals(person_a))
print("Age열:")
print(by_rows["Age"])
print("원래 age와 같은가?", by_rows["Age"].equals(age))
print("Series에서 값 선택:", person_a.loc["Age"])
print("표에서 값 선택:", by_rows.loc["A", "Age"])  # 둘 다 22.0

A행:
Age     22.00
Fare     7.25
Name: A, dtype: float64
원래 person_a와 같은가? True
Age열:
A    22.0
B    38.0
C    26.0
Name: Age, dtype: float64
원래 age와 같은가? True
Series에서 값 선택: 22.0
표에서 값 선택: 22.0


**정리:** 사람별 Series 3개를 행으로 놓아도, 항목별 Series 2개를 열로 놓아도 같은 3행 × 2열 표입니다. Series의 name은 전체 이름이고, index는 원소의 이름표입니다. 이름을 붙이는 것과 차원을 추가하는 것은 다릅니다.

<a id="n05"></a>
## N05 · 이름표를 기준으로 값을 맞추기

Series를 결합할 때 나열된 순서가 아니라 인덱스 레이블로 행을 맞춥니다.

### 요금의 순서를 뒤집거나 승객 일부를 빼 보기

기존 fare는 변경하지 않고 실험용 Series를 만듭니다.

In [ ]:
# N05
#Number index로 접근하기 iloc
#series인데 왜 [:,:,-1] 2차원처럼 접근하는가?
reversed_fare = fare.iloc[::-1]  # C, B, A 순서
print("순서를 뒤집은 요금:")
print(reversed_fare)
print("이름표로 다시 맞춘 표:")
print(pd.DataFrame({"Age": age, "Fare": reversed_fare}))

partial = pd.DataFrame({"Age": age, "Fare": fare.loc[["B", "C"]]})
print("A의 요금이 없는 경우:")
print(partial)  # A의 요금이 NaN: 연결할 값이 없음

순서를 뒤집은 요금:
C     7.93
B    71.28
A     7.25
Name: Fare, dtype: float64
이름표로 다시 맞춘 표:
    Age   Fare
A  22.0   7.25
B  38.0  71.28
C  26.0   7.93
A의 요금이 없는 경우:
    Age   Fare
A  22.0    NaN
B  38.0  71.28
C  26.0   7.93


<a id="n06"></a>
## N06 · concat으로 열을 옆에 붙이기

concat(concatenate, 이어 붙이다)은 기존 객체를 결합합니다. axis=1은 열이 늘어나는 방향입니다.

### Series 이름이 열 이름으로 쓰이는지 확인하기

N03·N04의 age와 fare를 그대로 사용합니다. ignore_index=True를 주면 결합하는 축의 기존 이름표를 버립니다.

In [13]:
# N06
joined_columns = pd.concat([age, fare], axis=1)
print(joined_columns)  # name인 Age, Fare가 열 이름으로 사용됨
print("딕셔너리 방식과 같은가?", joined_columns.equals(by_columns))
print("열 이름을 버린 경우:")
print(pd.concat([age, fare], axis=1, ignore_index=True))  # 열 이름이 0, 1
print("axis를 생략한 경우:")
print(pd.concat([age, fare]))  # 기본 axis=0: 값 6개인 Series, 두 열짜리 표가 아님

    Age   Fare
A  22.0   7.25
B  38.0  71.28
C  26.0   7.93
딕셔너리 방식과 같은가? True
열 이름을 버린 경우:
      0      1
A  22.0   7.25
B  38.0  71.28
C  26.0   7.93
axis를 생략한 경우:
A    22.00
B    38.00
C    26.00
A     7.25
B    71.28
C     7.93
dtype: float64


<a id="n07"></a>
## N07 · 행을 추가할 때 인덱스와 데이터 구분하기

두 표를 아래로 붙입니다. 행 인덱스의 중복과 데이터 내용의 중복은 서로 다릅니다.

### 0·0을 유지할지 0·1로 다시 매길지 비교하기

각 표에는 한 행뿐이므로 자동 행 인덱스가 각각 0입니다.

In [14]:
# N07
first = pd.DataFrame({"PassengerId": [101], "Age": [22]})
second = pd.DataFrame({"PassengerId": [102], "Age": [38]})
print("기존 행 인덱스 유지:")
print(pd.concat([first, second]))  # 0, 0도 허용됨
print("새 행 인덱스 부여:")
print(pd.concat([first, second], ignore_index=True))  # 0, 1
# PassengerId의 101, 102는 데이터 값이므로 바뀌지 않습니다.

기존 행 인덱스 유지:
   PassengerId  Age
0          101   22
0          102   38
새 행 인덱스 부여:
   PassengerId  Age
0          101   22
1          102   38


### 같은 행을 두 번 붙이면 왜 False·True일까?

duplicated는 기본적으로 앞에 같은 데이터가 있었는지 확인합니다. 행 인덱스는 중복 비교에 포함하지 않습니다.

In [ ]:
# N07
repeated = pd.concat([first, first], ignore_index=True)
print(repeated)
#중복된 object 찾기
print("뒤에 반복된 행:", repeated.duplicated().tolist())  # [False, True]
print("중복 그룹 전체:", repeated.duplicated(keep=False).tolist())  # [True, True]

   PassengerId  Age
0          101   22
1          101   22
뒤에 반복된 행: [False, True]
중복 그룹 전체: [True, True]


<a id="n08"></a>
## N08 · Series를 따로 만들지 않고 표 생성하기

딕셔너리 값으로 리스트를 전달하면 각각 하나의 열이 됩니다. 이 small 표를 N09~Q01에서 계속 사용합니다.

### 두 리스트를 Age·Fare 열로 만들기

같은 나이·요금 데이터를 더 짧게 만드는 방법입니다.

In [ ]:
# N08
small = pd.DataFrame({"Age": [22, 38, 26], "Fare": [7.25, 71.28, 7.93]},
                     index=["A", "B", "C"])
print(small)  # 정수만 있는 Age 열은 정수형으로 추론됨

   Age   Fare
A   22   7.25
B   38  71.28
C   26   7.93


<a id="n09"></a>
## N09 · 행 레이블과 행 위치 구분하기

index와 columns는 이름표, shape는 (행 수, 열 수)입니다. 정렬 후에도 기존 이름표는 유지됩니다.

### 나이순 정렬 후 이름표 확인하기

sort_values는 정렬 결과를 반환합니다. small 자체는 바꾸지 않습니다.

In [ ]:
# N09
print("행 레이블:", small.index.tolist())
print("열 레이블:", small.columns.tolist())
print("모양:", small.shape)
sorted_small = small.sort_values("Age")
print(sorted_small)
print("정렬 후 행 레이블:", sorted_small.index.tolist())  # A, C, B

행 레이블: ['A', 'B', 'C']
열 레이블: ['Age', 'Fare']
모양: (3, 2)
   Age   Fare
A   22   7.25
C   26   7.93
B   38  71.28
정렬 후 행 레이블: ['A', 'C', 'B']


<a id="n10"></a>
## N10 · 이름으로 고르기와 순서로 고르기

loc는 레이블 기준, iloc는 정수 위치 기준입니다. 위치는 0부터 셉니다.

### 같은 값과 범위를 두 방식으로 선택하기

loc의 레이블 슬라이스는 끝을 포함하고, iloc의 위치 슬라이스는 끝을 제외합니다.

In [ ]:
# N10
print("B의 나이:", small.loc["B", "Age"])
print("두 번째 행, 첫 번째 열:", small.iloc[1, 0])  # 둘 다 38
#["A":"B"] 이런식으로 범위 지정 가능
#시험
#이럴 때는 B 이전까지가 아니라 B포함까지임
print(small.loc["A":"B", ["Age", "Fare"]])  # B 포함
print(small.iloc[0:2, 0:2])  # 위치 2 제외
print("C, A 순서로 요금만:")
print(small.loc[["C", "A"], ["Fare"]])
print(small.iloc[[2, 0], [1]])

B의 나이: 38
두 번째 행, 첫 번째 열: 38
   Age   Fare
A   22   7.25
B   38  71.28
   Age   Fare
A   22   7.25
B   38  71.28
C, A 순서로 요금만:
   Fare
C  7.93
A  7.25
   Fare
C  7.93
A  7.25


<a id="n11"></a>
## N11 · 조건식으로 행 걸러내기

비교 결과는 행마다 True/False가 붙은 Series입니다. 이를 mask(선택 마스크)로 사용합니다.

### 조건 하나와 두 조건의 차이 보기

&는 두 조건을 모두 만족, |는 하나 이상 만족입니다. 각 조건을 괄호로 묶습니다.

In [ ]:
# N11
#bool 형식의 Series를 생성
mask = small["Age"] >= 25
print("25세 이상인가?")
print(mask)
print(small.loc[mask])  # B, C
print("25세 이상이면서 요금 10 미만:")
print(small.loc[(small["Age"] >= 25) & (small["Fare"] < 10)])  # C
print("25세 미만이거나 요금 50 초과:")
print(small.loc[(small["Age"] < 25) | (small["Fare"] > 50)])  # A, B

25세 이상인가?
A    False
B     True
C     True
Name: Age, dtype: bool
   Age   Fare
B   38  71.28
C   26   7.93
25세 이상이면서 요금 10 미만:
   Age  Fare
C   26  7.93
25세 미만이거나 요금 50 초과:
   Age   Fare
A   22   7.25
B   38  71.28


<a id="q01"></a>
## Q01 · 선택 결과 예상하기

① B의 요금 ② 처음 두 행·두 열 ③ 25세 이상이면서 요금 10 미만인 행을 선택하세요.

### 이름·위치·조건을 각각 적용한 풀이

출력은 순서대로 71.28, A·B행, C행입니다.

In [ ]:
# Q01
print("① B의 요금:", small.loc["B", "Fare"])
print("② 처음 두 행·두 열:")
print(small.iloc[:2, :2])
print("③ 두 조건을 만족하는 행:")
print(small.loc[(small["Age"] >= 25) & (small["Fare"] < 10)])

① B의 요금: 71.28
② 처음 두 행·두 열:
   Age   Fare
A   22   7.25
B   38  71.28
③ 두 조건을 만족하는 행:
   Age  Fare
C   26  7.93


<a id="n12"></a>
## N12 · Titanic 원자료 읽기

CSV(Comma-Separated Values)는 쉼표로 값을 구분한 파일입니다. raw는 가공 전 표를 보관하기 위해 정한 변수 이름입니다. 이후 실습에서 계속 사용합니다.

### 파일을 한 번 읽고 앞뒤 기록 살펴보기

한 행은 승객 한 명의 기록입니다. 이 자료는 전체 탑승객 명단이 아닌 891명의 공개 학습 자료입니다.

In [17]:
# N12
raw = pd.read_csv("./titanic_public.csv")
print("(행 수, 열 수):", raw.shape)  # (891, 12)
print(raw[["PassengerId", "Survived", "Pclass", "Age"]].head())
print(raw[["PassengerId", "Survived", "Pclass", "Age"]].head(3))
print("마지막 두 행:")
print(raw.tail(2))

(행 수, 열 수): (891, 12)
   PassengerId  Survived  Pclass   Age
0            1         0       3  22.0
1            2         1       1  38.0
2            3         1       3  26.0
3            4         1       1  35.0
4            5         0       3  35.0
   PassengerId  Survived  Pclass   Age
0            1         0       3  22.0
1            2         1       1  38.0
2            3         1       3  26.0
마지막 두 행:
     PassengerId  Survived  Pclass                   Name   Sex   Age  SibSp  \
889          890         1       1  Behr, Mr. Karl Howell  male  26.0      0   
890          891         0       3    Dooley, Mr. Patrick  male  32.0      0   

     Parch  Ticket   Fare Cabin Embarked  
889      0  111369  30.00  C148        C  
890      0  370376   7.75   NaN        Q  


<a id="n13"></a>
## N13 · 자료형·빈칸·분포 파악하기

info는 열의 자료형과 채워진 개수, describe는 수치 요약, value_counts는 값별 빈도입니다.

### 어떤 열에 무엇이 들어 있는지 확인하기

Survived는 생존 여부(0/1), Pclass는 객실 등급(1/2/3), Embarked는 승선 항구입니다.

In [6]:
# N13
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [ ]:
print(raw[["Age", "Fare"]].describe())
#count 결측치 뺀 갯수

              Age        Fare
count  714.000000  891.000000
mean    29.699118   32.204208
std     14.526497   49.693429
min      0.420000    0.000000
25%     20.125000    7.910400
50%     28.000000   14.454200
75%     38.000000   31.000000
max     80.000000  512.329200


In [18]:
print("\n", "성별 빈도:")
print(raw["Sex"].value_counts())
print("\n", "빈칸도 포함한 승선 항구 빈도:")
print(raw["Embarked"].value_counts(dropna=True)) 
print(raw["Embarked"].value_counts(dropna=False))  # 기본값은 결측 제외
print("\n", "나이 중앙값:", raw["Age"].median())  # 28


 성별 빈도:
Sex
male      577
female    314
Name: count, dtype: int64

 빈칸도 포함한 승선 항구 빈도:
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

 나이 중앙값: 28.0


<a id="n14"></a>
## N14 · 빈칸과 숫자 0 구분하기

isna(is NA)는 결측 여부를 판단합니다. NA는 값이 없거나 알려지지 않음을 나타내며 NaN(Not a Number)도 결측 표현으로 사용됩니다.

### 어느 열에 몇 개의 값이 없는가?

True를 합하면 결측 개수가 됩니다. 0이라는 요금은 결측값과 다릅니다.

In [19]:
# N14
missing = raw.isna().sum()
print(missing[missing > 0])  # Age 177, Cabin 687, Embarked 2
print(raw.loc[raw["Age"].isna(), ["PassengerId", "Age"]].head())
print("요금이 0인 행 수:", (raw["Fare"] == 0).sum())
print("요금이 결측인 행 수:", raw["Fare"].isna().sum())  # 0개

Age         177
Cabin       687
Embarked      2
dtype: int64
    PassengerId  Age
5             6  NaN
17           18  NaN
19           20  NaN
26           27  NaN
28           29  NaN
요금이 0인 행 수: 15
요금이 결측인 행 수: 0


<a id="n15"></a>
## N15 · 분석할 열을 고르고 이름 바꾸기

df는 DataFrame을 줄여 쓰는 관례적인 변수 이름이며 예약어가 아닙니다. raw는 보존하고 필요한 열만 작업용 df에 복사합니다.

### 필요한 9개 열만 남기고 Sex를 Gender로 바꾸기

cols는 columns(열들)를 뜻하도록 정한 변수 이름입니다. copy는 복사, rename은 이름 변경입니다.

In [21]:
# N15
cols = ["PassengerId", "Survived", "Pclass", "Sex", "Age",
        "SibSp", "Parch", "Fare", "Embarked"]
df = raw[cols].copy().rename(columns={"Sex": "Gender"})
print(df.head(3))
print("원자료 열:", raw.columns.tolist())
print("작업용 열:", df.columns.tolist())  # 원자료에는 여전히 Sex가 있음
print("제외할 열을 지정하는 방법:", (raw.drop(columns=["Name", "Ticket", "Cabin"]).shape))

   PassengerId  Survived  Pclass  Gender   Age  SibSp  Parch     Fare Embarked
0            1         0       3    male  22.0      1      0   7.2500        S
1            2         1       1  female  38.0      1      0  71.2833        C
2            3         1       3  female  26.0      0      0   7.9250        S
원자료 열: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
작업용 열: ['PassengerId', 'Survived', 'Pclass', 'Gender', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
제외할 열을 지정하는 방법: (891, 9)


<a id="n16"></a>
## N16 · 승객 조건 검색하기

N15의 df를 그대로 사용합니다. 열을 선택하거나 조건을 적용한 결과는 별도 변수에 저장합니다.

### 1등급 생존자와 나이가 기록된 미성년자 선택하기

나이 결측 승객은 18세 미만인지 판정할 수 없으므로 이 조건에 포함되지 않습니다.

In [20]:
# N16
selected = df.loc[(df["Survived"] == 1) & (df["Pclass"] == 1),
                  ["PassengerId", "Age", "Fare"]]
print(selected.head())
print("1등급 생존자 수:", len(selected))  # 136

children = df.loc[df["Age"] < 18, ["PassengerId", "Age", "Survived"]]
print(children.head())
print("나이가 기록된 18세 미만:", len(children))

    PassengerId   Age      Fare
1             2  38.0   71.2833
3             4  35.0   53.1000
11           12  58.0   26.5500
23           24  28.0   35.5000
31           32   NaN  146.5208
1등급 생존자 수: 136
    PassengerId   Age  Survived
7             8   2.0         0
9            10  14.0         1
10           11   4.0         1
14           15  14.0         0
16           17   2.0         0
나이가 기록된 18세 미만: 113


<a id="n17"></a>
## N17 · 정렬해서 상위 기록 보기

ascending=False는 내림차순입니다. 두 기준을 지정하면 첫 기준이 같은 행을 두 번째 기준으로 정렬합니다.

### 요금이 같은 승객은 식별번호순으로 정렬하기

정렬로 위치가 바뀌어도 행 인덱스는 유지됩니다. reset_index로 새 번호를 붙일 수 있습니다.

In [ ]:
# N17
#시험
ordered = df.sort_values(["Fare", "PassengerId"], ascending=[False, True]) # Fare : 내림차순, ID : 오름차순
print(ordered[["PassengerId", "Fare"]].head(5))

print("행 인덱스를 다시 붙인 경우:")
#재정렬된 dataframe에 number index 다시 붙이기
#Original index를 하나의 Series로 만들어버림
print(ordered[["PassengerId", "Fare"]].reset_index(drop=False).head(5))
print(ordered[["PassengerId", "Fare"]].reset_index(drop=False).head(5).shape)
print(ordered[["PassengerId", "Fare"]].reset_index(drop=True).head(5))
print(ordered[["PassengerId", "Fare"]].reset_index(drop=True).head(5).shape)
# drop=True는 예전 인덱스를 데이터 열로 남기지 않는다는 뜻입니다.

     PassengerId      Fare
258          259  512.3292
679          680  512.3292
737          738  512.3292
27            28  263.0000
88            89  263.0000
행 인덱스를 다시 붙인 경우:
   index  PassengerId      Fare
0    258          259  512.3292
1    679          680  512.3292
2    737          738  512.3292
3     27           28  263.0000
4     88           89  263.0000
(5, 3)
   PassengerId      Fare
0          259  512.3292
1          680  512.3292
2          738  512.3292
3           28  263.0000
4           89  263.0000
(5, 2)


<a id="q02"></a>
## Q02 · 조건 선택과 정렬 연결하기

1등급 생존자 중 요금이 높은 5명을 구하세요. 요금이 같으면 PassengerId 오름차순으로 정렬하고 PassengerId·Age·Fare를 출력하세요.

### 조건으로 고른 뒤 정렬하는 풀이

먼저 조건을 적용해야 다른 등급이나 사망한 승객이 섞이지 않습니다.

In [34]:
import pandas as pd

# 1. 1등급이면서 생존한 승객만 선택
selected = df.loc[(df["Pclass"] == 1) & (df["Survived"] == 1)]

# 2. 요금 내림차순, 요금이 같으면 승객 번호 오름차순
ordered = selected.sort_values(
    ["Fare", "PassengerId"],
    ascending=[False, True]
)

# 3. 필요한 세 열의 상위 5행을 answer에 저장
answer = ordered[["PassengerId", "Age", "Fare"]].head(5)

print(answer)

     PassengerId   Age      Fare
258          259  35.0  512.3292
679          680  36.0  512.3292
737          738  35.0  512.3292
88            89  23.0  263.0000
341          342  24.0  263.0000


<a id="n18"></a>
## N18 · 원자료와 작업용 표를 분리하는 이유

CSV 파일, 파일에서 읽은 raw, 복사본은 서로 구분합니다. 여기서는 비교용 표만 수정하며 N15의 df는 바꾸지 않습니다.

### 복사본을 바꾸어도 raw가 유지되는지 보기

0 대입은 복사 효과를 보여주기 위한 실험입니다. 실제 나이 결측값을 0으로 처리하라는 뜻이 아닙니다.

In [ ]:
# N18
copy_demo = raw.copy()
#Age의 모든 결측치 값을 0으로 변경
copy_demo.loc[copy_demo["Age"].isna(), "Age"] = 0
print("원자료의 나이 결측:", raw["Age"].isna().sum())  # 177
print("복사본의 나이 결측:", copy_demo["Age"].isna().sum())  # 0

alias = copy_demo  # 복사가 아니라 같은 객체에 이름을 하나 더 붙임
alias.loc[0, "Fare"] = 999
print("같은 객체인가?", alias is copy_demo)  # True
print("copy_demo의 요금:", copy_demo.loc[0, "Fare"])  # 999
print("raw의 요금:", raw.loc[0, "Fare"])  # 7.25, 유지됨

원자료의 나이 결측: 177
복사본의 나이 결측: 0
같은 객체인가? True
copy_demo의 요금: 999.0
raw의 요금: 7.25


<a id="n19"></a>
## N19 · 일부러 만든 중복 기록 찾기

N07의 concat을 활용해 첫 승객 기록을 한 번 더 붙입니다. 데이터 값의 중복을 확인합니다.

### 한 행을 추가하고 중복 제거 전후 비교하기

keep=False는 첫 기록까지 포함해 중복 그룹 전체를 표시합니다.

In [44]:
# N19
demo = pd.concat([raw, raw.iloc[[0]]], ignore_index=True)
print("추가 전:", len(raw), "/ 추가 후:", len(demo))  # 891 → 892
print("중복으로 판정된 행 수:", demo.duplicated().sum())  # 1
print(demo.loc[demo.duplicated(keep=False), ["PassengerId", "Age"]])
print("중복 제거 후:", len(demo.drop_duplicates()))  # 891
print("원자료로 돌아왔는가?", demo.drop_duplicates().equals(raw))

추가 전: 891 / 추가 후: 892
중복으로 판정된 행 수: 1
     PassengerId   Age
0              1  22.0
891            1  22.0
중복 제거 후: 891
원자료로 돌아왔는가? True


<a id="n20"></a>
## N20 · 중복 기준을 잘못 정하면 생기는 일

같은 나이·성별이라고 같은 사람은 아닙니다. 101번의 반복 기록과 서로 다른 102번 승객을 구분합니다.

### 나이·성별만 비교하면 누가 사라질까?

subset은 비교에 사용할 열 목록입니다. 이 예제의 세 번째 행은 첫 번째 행의 반복 기록입니다.

In [51]:
# N20
people = pd.DataFrame({"PassengerId": [101, 102, 101],
                       "Age": [22, 22, 22], "Sex": ["male", "male", "male"]})
print(people)
print("모든 열로 비교:", people.duplicated().tolist())  # False, False, True
print("나이·성별만 비교:", people.duplicated(subset=["Age", "Sex"]).tolist())
print("잘못된 기준으로 남은 승객:")
print(people.drop_duplicates(subset=["Age", "Sex"]))  # 102번까지 사라짐
print("모든 열을 비교한 결과:")
print(people.drop_duplicates())  # 101, 102 유지

   PassengerId  Age   Sex
0          101   22  male
1          102   22  male
2          101   22  male
모든 열로 비교: [False, False, True]
나이·성별만 비교: [False, True, True]
잘못된 기준으로 남은 승객:
   PassengerId  Age   Sex
0          101   22  male
모든 열을 비교한 결과:
   PassengerId  Age   Sex
0          101   22  male
1          102   22  male


<a id="n21"></a>
## N21 · 평균으로 나이 빈칸 채우기

clean은 정제 작업용 복사본입니다. N15의 df에서 출발하며 이후 계산한 열을 계속 추가합니다. 실제 나이를 알아낸 것이 아니라 정한 규칙으로 대체하는 것입니다.

### 평균 대체와 중앙값 대체가 어떻게 다른가?

AgeMissing에 원래 빈칸이던 위치를 남깁니다. mean은 평균, median은 중앙값, fillna는 결측값 채우기입니다.

In [58]:
# N21
clean = df.copy()
clean["AgeMissing"] = clean["Age"].isna()
mean_age = clean["Age"].mean()  # 결측값을 제외한 평균
clean["Age"] = clean["Age"].fillna(mean_age)
print("평균:", mean_age, "/ 중앙값:", df["Age"].median())

comparison = pd.DataFrame({"평균대체": clean["Age"],
                           "중앙값대체": df["Age"].fillna(df["Age"].median())})
print(comparison.loc[clean["AgeMissing"]].head())
print("대체 후 빈칸:", clean["Age"].isna().sum())  # 0

평균: 29.69911764705882 / 중앙값: 28.0
         평균대체  중앙값대체
5   29.699118   28.0
17  29.699118   28.0
19  29.699118   28.0
26  29.699118   28.0
28  29.699118   28.0
대체 후 빈칸: 0


<a id="n22"></a>
## N22 · 결측 행 삭제와 미상 범주 비교하기

결측값이 있다는 이유만으로 모든 행을 버리면 자료가 크게 줄어듭니다. 필요한 열을 기준으로 판단합니다.

### 항구 빈칸만 삭제할 때와 모든 빈칸을 삭제할 때 비교하기

원자료의 Cabin 결측이 많아서 모든 열을 검사하면 183명만 남습니다. clean에서는 삭제 대신 Unknown으로 표시합니다.

In [57]:
# N22
print("원자료:", len(raw))  # 891
print("승선 항구가 없는 행만 제외:", len(raw.dropna(subset=["Embarked"])))  # 889
print("어느 열이든 빈칸이 있으면 제외:", len(raw.dropna()))  # 183
print(clean["Embarked"].value_counts())
clean["Embarked"] = clean["Embarked"].fillna("Unknown")
print(clean["Embarked"].value_counts())  # Unknown 2명, 어느 항구인지 복원한 것은 아님

원자료: 891
승선 항구가 없는 행만 제외: 889
어느 열이든 빈칸이 있으면 제외: 183
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
Embarked
S          644
C          168
Q           77
Unknown      2
Name: count, dtype: int64


<a id="q03"></a>
## Q03 · 행 수를 유지하며 빈칸 처리하기

df를 복사하여 나이는 중앙값, 승선 항구는 Unknown으로 채우세요. clean의 평균 대체 결과와 별도로 보관합니다.

### 처리 전후 빈칸과 인원수를 출력하는 풀이

결측값이 0개가 되었는지와 승객 891명을 유지했는지를 함께 봅니다.

In [73]:
# Q03
practice = df.copy()
print(practice[["Age", "Embarked"]].isna().sum())
practice["Age"] = practice["Age"].fillna(practice["Age"].median())
practice["Embarked"] = practice["Embarked"].fillna("Unknown")
print("승객 수:", len(practice))
print("처리 후 빈칸:")
print(practice[["Age", "Embarked"]].isna().sum())

Age         177
Embarked      2
dtype: int64
승객 수: 891
처리 후 빈칸:
Age         0
Embarked    0
dtype: int64


<a id="n23"></a>
## N23 · 계산한 열과 범주 코드를 추가하기

SibSp는 siblings/spouses(형제자매·배우자 수), Parch는 parents/children(부모·자녀 수)입니다. Gender는 N15에서 Sex를 바꾼 열입니다.

### 본인을 포함한 가족 수와 성별 코드 만들기

map은 대응표로 값을 바꿉니다. 0·1은 여기서 지정한 코드이지 성별의 크기나 순서가 아닙니다.

In [ ]:
# N23
clean["FamilySize"] = clean["SibSp"] + clean["Parch"] + 1
#male을 0으로, female을 1 치환하겠다
clean["GenderCode"] = clean["Gender"].map({"male": 0, "female": 1})
print(clean[["SibSp", "Parch", "FamilySize", "Gender", "GenderCode"]].head(3))



   SibSp  Parch  FamilySize  Gender  GenderCode
0      1      0           2    male           0
1      1      0           2  female           1
2      0      0           1  female           1


In [75]:
print("대응표에 없는 값은?")
print(pd.Series(["male", "female", "unknown"]))
print(pd.Series(["male", "female", "unknown"]).map({"male": 0, "female": 1}))
# unknown은 대응하는 값이 없으므로 NaN이 됩니다.

대응표에 없는 값은?
0       male
1     female
2    unknown
dtype: object
0    0.0
1    1.0
2    NaN
dtype: float64


<a id="n24"></a>
## N24 · 큰 값을 지우기 전에 기록 확인하기

값이 크다는 사실만으로 입력 오류라고 단정하지 않습니다. 승객 번호와 티켓을 함께 확인합니다.

### 500 이상의 요금을 낸 기록 찾기

이 실습은 확인만 하고 승객을 삭제하지 않습니다.

In [76]:
# N24
high_fare = raw.loc[raw["Fare"] >= 500, ["PassengerId", "Ticket", "Fare"]]
print(high_fare)  # 3명 모두 요금 512.3292
print(raw["Fare"].describe(percentiles=[0.5, 0.95, 0.99]))
print("유지한 승객 수:", len(clean))  # 891

     PassengerId    Ticket      Fare
258          259  PC 17755  512.3292
679          680  PC 17755  512.3292
737          738  PC 17755  512.3292
count    891.000000
mean      32.204208
std       49.693429
min        0.000000
50%       14.454200
95%      112.079150
99%      249.006220
max      512.329200
Name: Fare, dtype: float64
유지한 승객 수: 891


<a id="n25"></a>
## N25 · 값의 범위를 0~1로 바꾸기

최소·최대 정규화는 (값−최솟값)/(최댓값−최솟값)입니다. 거리 기반 분석 등의 전처리에 쓰지만 오늘의 생존율 계산에는 필요하지 않습니다.

### 작은 숫자로 원리를 보고 요금에 적용하기

원래 Fare는 남기고 FareScaled라는 새 열에 결과를 저장합니다.

In [77]:
# N25
toy = pd.Series([10, 20, 30])
print((toy - toy.min()) / (toy.max() - toy.min()))  # 0, 0.5, 1
fare_values = clean["Fare"]
clean["FareScaled"] = (fare_values - fare_values.min()) / (fare_values.max() - fare_values.min())
print(clean[["Fare", "FareScaled"]].head(3))

0    0.0
1    0.5
2    1.0
dtype: float64
      Fare  FareScaled
0   7.2500    0.014151
1  71.2833    0.139136
2   7.9250    0.015469


### 모든 값이 같아서 분모가 0이면?

그대로 나누지 않고 이 예제에서는 모두 0으로 두는 규칙을 정합니다.

In [78]:
# N25
constant = pd.Series([5, 5, 5])
span = constant.max() - constant.min()
if span == 0:
    scaled = pd.Series(0.0, index=constant.index)
else:
    scaled = (constant - constant.min()) / span
print("분모:", span)
print(scaled)

분모: 0
0    0.0
1    0.0
2    0.0
dtype: float64


<a id="q04"></a>
## Q04 · 계산 결과를 읽고 설명하기

기록된 동승 가족이 없는 승객 수, 성별 코드의 종류와 결측 개수, 정규화 범위, 원래 요금 보존 여부를 출력하세요. N21~N25의 clean을 그대로 사용합니다.

### 새 열의 의미와 원자료 보존을 확인하는 풀이

오류 없이 실행되는 것과 의미 있는 결과를 얻는 것은 다릅니다. 출력된 수치를 설명해 보세요.

In [80]:
# Q04
print("기록된 동승 가족이 없는 승객:", (clean["FamilySize"] == 1).sum())  # 537
print("성별 코드 종류:", clean["GenderCode"].unique())  # 0과 1
print("성별 코드 빈칸:", clean["GenderCode"].isna().sum())  # 0
print("정규화 범위:", clean["FareScaled"].min(), clean["FareScaled"].max())  # 0, 1
print("원래 요금 보존:", clean["Fare"].equals(raw["Fare"]))  # True

기록된 동승 가족이 없는 승객: 537
성별 코드 종류: [0 1]
성별 코드 빈칸: 0
정규화 범위: 0.0 1.0
원래 요금 보존: True


<a id="n26"></a>
## N26 · 그룹별 생존율 계산하기

groupby는 기준이 같은 행을 묶고, agg(aggregate, 집계)는 묶음마다 계산합니다. 생존 1·사망 0의 평균은 생존자 비율입니다.

### 등급별 인원수·생존자 수·생존율 함께 보기

size는 행 수, sum은 생존자 수, mean은 생존율입니다. 생존율은 전체 승객이 아니라 이 자료의 승객을 기준으로 합니다.

In [ ]:
#시험
print(clean.groupby("Pclass"))
groups = clean.groupby("Pclass")

#get_group
print("1등급 승객:")
print(groups.get_group(1)[["PassengerId", "Pclass", "Survived"]])

print("2등급 승객:")
print(groups.get_group(2)[["PassengerId", "Pclass", "Survived"]])

print("3등급 승객:")
print(groups.get_group(2)[["PassengerId", "Pclass", "Survived"]])

1등급 승객:
     PassengerId  Pclass  Survived
1              2       1         1
3              4       1         1
6              7       1         0
11            12       1         1
23            24       1         1
..           ...     ...       ...
871          872       1         1
872          873       1         0
879          880       1         1
887          888       1         1
889          890       1         1

[216 rows x 3 columns]
2등급 승객:
     PassengerId  Pclass  Survived
9             10       2         1
15            16       2         1
17            18       2         1
20            21       2         0
21            22       2         1
..           ...     ...       ...
866          867       2         1
874          875       2         1
880          881       2         1
883          884       2         0
886          887       2         0

[184 rows x 3 columns]
3등급 승객:
     PassengerId  Pclass  Survived
9             10       2         1
15            16  

In [ ]:
# N26
#시험
#.agg
summary = clean.groupby("Pclass")["Survived"].agg(
    passengers="size", survivors="sum", survival_rate="mean"
)
print(summary)


        passengers  survivors  survival_rate
Pclass                                      
1              216        136       0.629630
2              184         87       0.472826
3              491        119       0.242363


In [98]:
# o = 생존, x = 사망이라고 가정하면 agg를 사용하기 위해서 Map 적용
clean["SurvivedCode"] = clean["Survived"].map({"o": 1, "x": 0})

summary = clean.groupby("Pclass")["SurvivedCode"].agg(
    passengers="size",
    survivors="sum",
    survival_rate="mean"
)

In [97]:
print("전체 인원:", summary["passengers"].sum())  # 891
print("전체 생존자:", summary["survivors"].sum())  # 342
print("성별로 묶기:")
print(clean.groupby("Gender")["Survived"].agg(["size", "mean"]))
print("등급과 성별을 함께 기준으로 묶기:")
print(clean.groupby(["Pclass", "Gender"])["Survived"].agg(["size", "mean"]))

전체 인원: 891
전체 생존자: 342
성별로 묶기:
        size      mean
Gender                
female   314  0.742038
male     577  0.188908
등급과 성별을 함께 기준으로 묶기:
               size      mean
Pclass Gender                
1      female    94  0.968085
       male     122  0.368852
2      female    76  0.921053
       male     108  0.157407
3      female   144  0.500000
       male     347  0.135447


<a id="e01"></a>
## E01 · 처음부터 결과 표까지 한 번에 연결하기

CSV 읽기 → 열 선택·이름 변경 → 중복 확인 → 결측값 처리 → 새 열 계산 → 등급별 집계를 한 번에 수행하세요. 아래는 전체 풀이입니다. 이 종합 실습에서만 앞 코드를 한 번 모아 반복합니다.

### 전체 흐름을 처음부터 실행하는 풀이

다른 실습 변수가 바뀌어도 이 셀은 입력부터 다시 준비합니다. 처리마다 무엇이 바뀌었는지 출력합니다.

In [101]:
# E01
import pandas as pd

source = pd.read_csv("/content/titanic_public.csv")
columns = ["PassengerId", "Survived", "Pclass", "Sex", "Age",
           "SibSp", "Parch", "Fare", "Embarked"]
result = source[columns].copy().rename(columns={"Sex": "Gender"})
print("원래 승객 수:", len(result), "/ 중복 행 수:", result.duplicated().sum())
result = result.drop_duplicates()  # 이 파일에는 중복이 없어 행 수가 유지됨

# 실제 나이를 복원한 것이 아니라 평균으로 대체하고 대체 위치를 남깁니다.
result["AgeMissing"] = result["Age"].isna()
result["Age"] = result["Age"].fillna(result["Age"].mean())
result["Embarked"] = result["Embarked"].fillna("Unknown")
print("나이를 채운 인원:", result["AgeMissing"].sum())

result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
result["GenderCode"] = result["Gender"].map({"male": 0, "female": 1})
fare_values = result["Fare"]
span = fare_values.max() - fare_values.min()
result["FareScaled"] = (fare_values - fare_values.min()) / span if span != 0 else 0.0

final_summary = result.groupby("Pclass")["Survived"].agg(
    passengers="size", survivors="sum", survival_rate="mean"
)
print("완성된 표의 모양:", result.shape)  # 891행 × 13열
print("남은 빈칸 수:", result.isna().sum().sum())  # 0
print(final_summary)
# 정제는 질문에 맞춰 선택합니다. 모든 분석에 모든 처리가 필요한 것은 아닙니다.

원래 승객 수: 891 / 중복 행 수: 0
나이를 채운 인원: 177
완성된 표의 모양: (891, 13)
남은 빈칸 수: 0
        passengers  survivors  survival_rate
Pclass                                      
1              216        136       0.629630
2              184         87       0.472826
3              491        119       0.242363


<a id="n27"></a>
## N27 · 결과를 파일로 저장하기

to_csv는 to CSV, 즉 CSV로 내보내기입니다. Path는 경로, mkdir(make directory)은 폴더 생성입니다.

### 승객 표와 요약표를 저장하고 다시 읽기

N26·E01에서 만든 결과를 재사용합니다. index=False는 행 인덱스를 별도 데이터 열로 저장하지 않는 설정입니다.

In [102]:
# N27
Path("results").mkdir(exist_ok=True)  # 폴더가 이미 있어도 오류 내지 않음
result.to_csv("results/titanic_clean.csv", index=False)
final_summary.reset_index().to_csv("results/survival_by_class.csv", index=False)

# Pclass는 집계 표의 인덱스이므로 일반 열로 옮겨서 저장합니다.
rate_table = summary[["survival_rate"]].reset_index()
rate_table.to_csv("results/survival_rate.csv", index=False)
saved = pd.read_csv("results/survival_rate.csv")
print(saved)
print("생존율(%):")
print(saved["survival_rate"].mul(100).round(1))  # mul(multiply): 곱하기

   Pclass  survival_rate
0       1            NaN
1       2            NaN
2       3            NaN
생존율(%):
0   NaN
1   NaN
2   NaN
Name: survival_rate, dtype: float64
